# 11 — Givens & Extensions

Notebook 10 gave you generics — the ability to abstract over types. This notebook gives you the partner mechanism: a way for the compiler to *provide* values for those types automatically, based on type alone. That's what **givens** are. Combined with **extension methods**, they unlock the type-class style of programming — adding capabilities to types without modifying their source.

If you've used Scala 2, this is the modern rework of `implicit val`, `implicit def`, and `implicit class`. The mechanism is the same. The keywords are clearer, the failure modes are more constrained, and the resulting code reads more like intent. If you haven't seen the implicits world before, even better — start fresh with the Scala 3 names.

The plan:

1. **Givens** — declare a value the compiler can find by its type.
2. **`using` parameters** — receive a given without naming it at the call site.
3. **Type classes** — the design pattern that gives this its power.
4. **Extension methods** — add methods to types you don't own.
5. **Implicit conversions** — the controlled descendant of Scala 2's `implicit def`, used sparingly.

## The motivation — a dependency you keep passing around

Most non-trivial functions take *two* kinds of inputs: the data they operate on, and contextual values they need to do the job — a `Logger`, an `ExecutionContext`, a database connection, a comparison strategy. Pass the contextual values explicitly through every call and your signatures bloat. Reach for global singletons and you lose testability.

Givens are a middle path. You *declare* the contextual value somewhere visible, mark the receiving parameter with `using`, and the compiler threads the value through for you. The dependency stays in the type signature — so the function is still honest about what it needs — but you stop typing it at every call.

## Givens — values the compiler can find by type

A `given` declaration is an ordinary value with one extra property: the compiler indexes it by its type and can look it up automatically when something asks for that type. The simplest form is a one-liner that names a value and its type.

In [ ]:
given defaultGreeting: String = "hello"

// The given is in scope. Anything that asks for a String via `using` finds it.
// The name `defaultGreeting` is mostly for documentation — lookup is by type.

Notice the name is optional. If you write `given String = "hello"` the compiler invents one. The name only matters if you need to refer to the given explicitly (for instance to import it). What matters for resolution is the *type*.

## `using` parameters — receiving a given

A parameter list introduced with the `using` keyword tells the compiler: *don't make me pass these explicitly; find them by type in the enclosing scope.*

In [ ]:
given defaultGreeting: String = "hello"

def greet(name: String)(using greeting: String): String =
  s"$greeting, $name"

greet("alice")                       // "hello, alice" — given supplied automatically
greet("bob")(using "howdy")          // "howdy, bob"   — override at the call site

Two things to notice. First, the `using` parameter list is a separate parameter list — it comes *after* the regular arguments. Second, you can always pass it explicitly with `(using value)` at the call site if you want to override the default. The mechanism is opt-in inference, not magic.

## `summon` — asking for a given by type

Sometimes you need to fetch a given inside a function body without declaring a `using` parameter for it. The built-in `summon[T]` does exactly that — *give me whatever value of type `T` is in given scope*.

In [ ]:
given defaultGreeting: String = "hello"

val current: String = summon[String]   // "hello"
// Equivalent to the more verbose: implicitly[String] in Scala 2

`summon` is rarely needed in everyday code — `using` parameters cover the common case. It's most useful inside generic helpers and inside `given` definitions that depend on other givens. We'll see it in action below.

## Why this isn't just global state

A reasonable first reaction is: *isn't this just dressed-up globals?* It isn't, for two reasons.

- **Type-directed.** Lookup is by *type*, not by name. So you can have a `given Logger`, a `given Clock`, and a `given Config` in the same scope without collision — the compiler picks the one matching the type each `using` asks for.
- **Scoped.** Givens are values bound to the scope they're declared in. They participate in normal import rules. If you don't import the given, it isn't available — which means tests can swap in their own without rewriting production code.

These two properties are what turn a small language feature into the foundation of *type classes*, which is the next stop.

## Type classes — the pattern that makes givens shine

A **type class** is a way to attach behaviour to a type *from the outside*, without modifying that type's source. In Java or Python, the equivalent is making the type implement an interface — but that requires owning the type. Type classes work even for types you don't own (like `Int`, `String`, or someone else's library class).

The recipe has three steps. We'll build a small `Show` type class that gives any type a `show` method — like `toString`, but type-safe and instance-by-instance.

### Step 1 — the trait

Declare a generic trait describing the capability. One abstract method, parameterised by the type you want to add behaviour to.

In [ ]:
trait Show[A]:
  def show(a: A): String

### Step 2 — instances as givens

For each concrete type you want to support, provide a `given` that supplies an implementation. These are the *type-class instances*.

In [ ]:
given Show[Int] with
  def show(a: Int): String = a.toString

given Show[String] with
  def show(a: String): String = s"\"$a\""

given Show[Boolean] with
  def show(a: Boolean): String = if a then "yes" else "no"

The `with` syntax introduces an anonymous given whose body provides the trait's methods. Each one is a separate value indexed by its full type — `Show[Int]`, `Show[String]`, `Show[Boolean]`. They live as siblings; none of them collide because they have different parameter types.

### Step 3 — polymorphic code that uses the instances

Now write a generic function that says, *I work for any `A`, as long as there's a `Show[A]` available.*

In [ ]:
def render[A](a: A)(using s: Show[A]): String = s.show(a)

render(42)        // "42"        — picks given Show[Int]
render("scala")   // "\"scala\"" — picks given Show[String]
render(true)      // "yes"       — picks given Show[Boolean]

What's happening at each call: the compiler sees `render(42)`, infers `A = Int`, then looks for a `given Show[Int]` in scope, finds the one we declared, and threads it in. From your code's perspective, you wrote one polymorphic function and got per-type behaviour for free.

### Context bounds — the shorthand `[A: Show]`

Asking for an unnamed `using` parameter of type `F[A]` is so common that Scala has a one-character shorthand. `[A: Show]` means *there's some `A`, and there must be a `given Show[A]` available.* Inside the body, you fetch it with `summon`.

In [ ]:
def render[A: Show](a: A): String = summon[Show[A]].show(a)

render(42)        // "42"
render("scala")   // "\"scala\""

The two forms `def f[A](x: A)(using Show[A])` and `def f[A: Show](x: A)` are equivalent. Use the long form when you need to name the instance for repeated use; use the context bound when you only need it once via `summon`.

## Extension methods — adding methods to types you don't own

Type classes give you *functions* that work over many types. But the call syntax is `render(x)`, not `x.render`. Sometimes the second reads better — especially when chaining. Extension methods let you attach a method to an existing type from outside, without modifying its source.

In [ ]:
extension (s: String)
  def shout: String = s.toUpperCase + "!"
  def words: List[String] = s.split("\\s+").toList

"hello".shout            // "HELLO!"
"one two three".words    // List("one", "two", "three")

The `extension (s: String)` header introduces a *receiver* `s` of type `String`. Every `def` inside the block becomes a method callable on any `String`. The compiler rewrites `"hello".shout` into the equivalent function call under the hood — no magic, just sugar.

### Extensions on generic types

Extensions can take their own type parameters and even depend on givens. This is the form you'll use most when designing type-class APIs.

In [ ]:
extension [A](a: A)(using s: Show[A])
  def show: String = s.show(a)

42.show         // "42"
"scala".show    // "\"scala\""
true.show       // "yes"

Now `a.show` is just the method-call form of `render(a)` — same lookup, same behaviour, friendlier syntax. This is exactly how libraries like Cats give you `x === y`, `x |+| y`, and `xs.traverse(...)` as if they were native methods on every type.

## Composing givens — instances that depend on other instances

Givens can take `using` parameters of their own. The compiler chains them: ask for a `Show[List[Int]]`, and if there's a `given Show[Int]` plus a `given Show[List[A]] given Show[A]`, the compiler assembles the instance for you.

In [ ]:
given listShow[A](using s: Show[A]): Show[List[A]] with
  def show(as: List[A]): String =
    as.map(s.show).mkString("[", ", ", "]")

render(List(1, 2, 3))             // "[1, 2, 3]"
render(List("a", "b"))            // "[\"a\", \"b\"]"
render(List(List(1), List(2,3)))  // "[[1], [2, 3]]"

Read the last call carefully — the compiler resolves `Show[List[List[Int]]]` by stacking the `listShow` instance twice on top of `Show[Int]`. You wrote two givens; you got behaviour for every nested list type for free. This compositionality is what makes type classes scale to large libraries.

## Where to declare givens

Two locations matter, and the compiler looks in both automatically:

- **The companion object of the type class.** Put your standard instances on `object Show` and they're found whenever someone asks for any `Show[A]`. No import needed.
- **The companion object of the data type.** Put `given Show[Order]` on `object Order` and it's found whenever someone asks for `Show[Order]`. No import needed either.

Anything declared anywhere else needs an explicit `import` to come into given scope at the call site. The standard import for given values is `import obj.given` (all givens) or `import obj.{given Show[Int]}` (a specific one).

## You've already used type classes in the standard library

Two examples you've probably met without realising:

In [ ]:
// 1. Ordering — the type class behind sorting and comparison
List(3, 1, 2).sorted                       // List(1, 2, 3)
List("banana", "apple").sorted             // List("apple", "banana")

// The signature is roughly:
//   def sorted[B >: A](using ord: Ordering[B]): List[A]
// Different types sort differently because each provides its own given Ordering.

// 2. Custom orderings via `using`
given Ordering[String] = Ordering.by(_.length)
List("aaa", "b", "cc").sorted              // List("b", "cc", "aaa")

Every time you call `.sorted`, `.min`, `.max`, or `.sum`, the compiler is silently supplying an `Ordering` or `Numeric` instance from given scope. That's why `List(1,2,3).sum` works and `List(Some(1), Some(2)).sum` doesn't — there's no `Numeric[Option[Int]]` in scope.

## Implicit conversions — the controlled descendant

Scala 3 keeps automatic value conversions, but locks them behind an explicit type: `Conversion[From, To]`. Declaring one as a `given` lets the compiler insert it where a `From` is supplied but a `To` is expected.

In [ ]:
import scala.language.implicitConversions

given Conversion[Int, String] = _.toString

val s: String = 42    // legal — compiler inserts the conversion

The `import scala.language.implicitConversions` is required at the call site — Scala 3 makes you opt in deliberately. That import is the visible warning sign in a code review.

**Use these sparingly.** Most of the time when you think you want a conversion, you actually want an extension method or an explicit `.to` call. Conversions blur the type system: a `String` parameter quietly accepting an `Int` is exactly the kind of thing that turns into a long bug hunt years later. Reserve them for true equivalence cases (numeric widening, library interop) where the conversion is genuinely information-preserving.

## Common pitfalls

A short list of the failure modes you'll hit at first:

- **"No given instance of type X was found."** The compiler couldn't find a value to fill the `using`. Either declare one, or import it from where it lives. The error always names the exact type it was looking for.
- **Ambiguous givens.** If two `given Show[Int]` are in scope, the compiler refuses to choose. Move one out, or import only the one you want.
- **Forgetting `with` or `=` in a given definition.** `given Show[Int] with def show(a: Int) = ...` and `given Show[Int] = new Show[Int]: ...` are both valid; missing the connector is a common typo.
- **Putting instances in a random file.** Givens that aren't in a companion object only count when explicitly imported. A package-level `given` in a deep file is easy to miss.

## Putting it together — a small JSON encoder

A realistic miniature: define a `JsonEncoder` type class, provide instances for primitives, derive an instance for `List`, and add a method-style syntax.

In [ ]:
trait JsonEncoder[A]:
  def encode(a: A): String

object JsonEncoder:
  given JsonEncoder[Int]     with def encode(a: Int): String     = a.toString
  given JsonEncoder[String]  with def encode(a: String): String  = s"\"$a\""
  given JsonEncoder[Boolean] with def encode(a: Boolean): String = a.toString

  given listEncoder[A](using e: JsonEncoder[A]): JsonEncoder[List[A]] with
    def encode(as: List[A]): String =
      as.map(e.encode).mkString("[", ",", "]")

extension [A](a: A)(using e: JsonEncoder[A])
  def toJson: String = e.encode(a)

42.toJson                              // "42"
"hello".toJson                         // "\"hello\""
List(1, 2, 3).toJson                   // "[1,2,3]"
List(List("a"), List("b", "c")).toJson // "[[\"a\"],[\"b\",\"c\"]]"

Read that example end-to-end and notice the layering. The `trait` describes the capability. Instances live on the *companion object* of the type class, so they're found automatically. The `listEncoder` given takes another given as a `using` parameter, which lets the compiler derive encoders for lists of anything that already has one. The extension method gives you the `.toJson` syntax. None of `Int`, `String`, or `List` knows anything about JSON — yet they all gain the method through the type class machinery.

## What's next

Notebook 12 covers **advanced types** — opaque types, type aliases, union and intersection types, match types, and a closer look at variance and bounds. Together with this notebook's givens and extensions, those are the tools that let Scala libraries express precise constraints in types while keeping the day-to-day usage syntax clean.